# Sanday Colab Experiment

Use this notebook with a Google Colab GPU kernel from VS Code. It prepares the repository, installs dependencies, checks the dataset, runs a short smoke test, trains, evaluates, and shows the saved artifacts.

In [8]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/karl4th/sanday.git'
COLAB_REPO = Path('/content/sanday')

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'configs').is_dir() and (candidate / 'scripts').is_dir() and (candidate / 'sanday').is_dir():
            return candidate
    if COLAB_REPO.exists():
        subprocess.run(['git', '-C', str(COLAB_REPO), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(COLAB_REPO)], check=True)
    return COLAB_REPO.resolve()

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)

Repo root: /content/sanday


In [22]:
!cd sanday && git pull --ff-only

remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 10 (delta 8), reused 10 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 1.36 KiB | 348.00 KiB/s, done.
From https://github.com/karl4th/sanday
   bf67fa2..e8559cd  main       -> origin/main
Updating bf67fa2..e8559cd
Fast-forward
 configs/sanday_sliding_ncp.yaml |  2 +-
 configs/sanday_smoke.yaml       | 80 +++++++++++++++++++++++++++++++++++++++++
 sanday/data.py                  |  3 ++
 sanday/features.py              |  2 +-
 scripts/train.py                | 12 +++++++
 5 files changed, 97 insertions(+), 2 deletions(-)
 create mode 100644 configs/sanday_smoke.yaml


## Configuration

In [10]:
CONFIG = 'configs/sanday_sliding_ncp.yaml'
RUN_NAME = 'sliding_ncp_seed42'
RUN_DIR = f'results/{RUN_NAME}'
SMOKE_DIR = f'{RUN_DIR}_smoke'
EVAL_DIR = f'{RUN_DIR}_eval'

RUN_SMOKE_TEST = True
TRAIN = True
EVALUATE = True
CHECKPOINT = f'{RUN_DIR}/checkpoints/sanday_best.pt'

print('Config:', CONFIG)
print('Run dir:', RUN_DIR)
print('Smoke dir:', SMOKE_DIR)
print('Checkpoint:', CHECKPOINT)

Config: configs/sanday_sliding_ncp.yaml
Run dir: results/sliding_ncp_seed42
Smoke dir: results/sliding_ncp_seed42_smoke
Checkpoint: results/sliding_ncp_seed42/checkpoints/sanday_best.pt


## Install Dependencies

In [19]:
%pip install -q -r requirements.txt

## Environment Check

In [12]:
import subprocess
import torch

print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('CUDA version:', torch.version.cuda)
print('Torch version:', torch.__version__)
print()
try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except Exception as exc:
    print('nvidia-smi failed:', exc)

CUDA: True
GPU: Tesla T4
CUDA version: 12.8
Torch version: 2.11.0+cu128

Thu Jun 18 11:00:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        

## Dataset Check

In [13]:
from pathlib import Path
import kagglehub

dataset_path = kagglehub.dataset_download('prateeknarain/common-voice')
print('Dataset path:', dataset_path)

root = Path(dataset_path)
english_validated = [p for p in root.rglob('validated.tsv') if '/English/' in p.as_posix() or '/en/' in p.as_posix()]
english_validated = sorted(english_validated, key=lambda p: (len(p.parts), str(p)))
print('English validated.tsv candidates:')
for item in english_validated[:10]:
    print(' -', item.relative_to(root))
if not english_validated:
    raise RuntimeError('No English validated.tsv found in dataset archive')

Using Colab cache for faster access to the 'common-voice' dataset.
Dataset path: /kaggle/input/common-voice
English validated.tsv candidates:
 - Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14/en/validated.tsv
 - Common Voice/Common Voice/English/cv-corpus-21.0-delta-2025-03-14-en/14765303730 14765303730/cv-corpus-21.0-delta-2025-03-14/en/validated.tsv


## Helper

In [15]:
import shlex
import subprocess
import sys
from collections import deque

def run_command(cmd, run_dir=None):
    print('Running:', ' '.join(shlex.quote(str(part)) for part in cmd), flush=True)
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = deque(maxlen=300)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        tail.append(line)
    return_code = process.wait()
    if return_code != 0:
        print('\nCommand failed with return code:', return_code)
        if run_dir is not None:
            error_log = Path(run_dir) / 'error.log'
            if error_log.exists():
                print('\n--- error.log ---')
                print(error_log.read_text()[-12000:])
            else:
                print('No error.log found at:', error_log)
                print('\n--- process output tail ---')
                print(''.join(tail)[-12000:])
        raise RuntimeError('Command failed')
    return return_code

## Smoke Test

In [ ]:
if RUN_SMOKE_TEST:
    cmd = [
        sys.executable, 'scripts/train.py',
        '--config', 'configs/sanday_smoke.yaml',
        '--run-dir', SMOKE_DIR,
        '--epochs', '1',
        '--max-train-batches', '2',
        '--max-valid-batches', '1',
    ]
    run_command(cmd, SMOKE_DIR)
else:
    print('RUN_SMOKE_TEST is False; skipping smoke test.')

## Train

In [ ]:
if TRAIN:
    cmd = [sys.executable, 'scripts/train.py', '--config', CONFIG, '--run-dir', RUN_DIR]
    run_command(cmd, RUN_DIR)
else:
    print('TRAIN is False; skipping training.')

## Training Results

In [ ]:
import json
import pandas as pd
from IPython.display import display

summary_path = Path(RUN_DIR) / 'summary.json'
metrics_path = Path(RUN_DIR) / 'metrics.csv'

if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
else:
    print('No summary found:', summary_path)

if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics.tail(10))
    display(metrics[['epoch', 'train_loss', 'valid_wer', 'valid_cer']].plot(x='epoch', grid=True, figsize=(10, 4)))
else:
    print('No metrics found:', metrics_path)

## Evaluate Best Checkpoint

In [ ]:
if EVALUATE:
    checkpoint_path = Path(CHECKPOINT)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')
    cmd = [sys.executable, 'scripts/evaluate.py', '--config', CONFIG, '--checkpoint', str(checkpoint_path), '--run-dir', EVAL_DIR]
    run_command(cmd, EVAL_DIR)
else:
    print('EVALUATE is False; skipping evaluation.')

## Evaluation Results

In [ ]:
evaluation_path = Path(EVAL_DIR) / 'evaluation.json'
predictions_path = Path(EVAL_DIR) / 'predictions.csv'

if evaluation_path.exists():
    evaluation = json.loads(evaluation_path.read_text())
    print(json.dumps(evaluation, indent=2))
else:
    print('No evaluation found:', evaluation_path)

if predictions_path.exists():
    predictions = pd.read_csv(predictions_path)
    display(predictions.head(20))
else:
    print('No predictions found:', predictions_path)

## Files To Share

In [ ]:
important_files = [
    Path(RUN_DIR) / 'config.yaml',
    Path(RUN_DIR) / 'environment.json',
    Path(RUN_DIR) / 'metrics.csv',
    Path(RUN_DIR) / 'metrics.jsonl',
    Path(RUN_DIR) / 'summary.json',
    Path(EVAL_DIR) / 'environment.json',
    Path(EVAL_DIR) / 'evaluation.json',
    Path(EVAL_DIR) / 'predictions.csv',
]

for path in important_files:
    print(('OK   ' if path.exists() else 'MISS '), path)

print('\nCheckpoint is intentionally large and ignored by git:')
print(Path(CHECKPOINT))